# RL Day 2 — Jupyter Notebook
**Heli → Fighter**

> This notebook runs on Binder. Your session is temporary — download any results you want to keep.

Today's games have continuous states. Watch the animation first, then adjust the **🔧** parameters.

In [ ]:
# ── Setup: download RR environments and import ──────────────────
RR_ENVS_URL = 'https://raw.githubusercontent.com/colombo0718/rl-lab-group-b/master/rr_envs.py'

!rm -f rr_envs.py
!wget -q "{RR_ENVS_URL}" -O rr_envs.py
!pip install gymnasium matplotlib numpy -q

import importlib, sys, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

if 'rr_envs' in sys.modules:
    del sys.modules['rr_envs']

from rr_envs import (
    MABEnv, Maze1DEnv, Maze2DEnv, HeliEnv, FighterEnv,
    show_env, animate_actions, animate_random, run_q_learning, plot_training, plot_maze2d_qtable, plot_maze1d_qtable,
)

def make_key_fn(env, bins=None):
    from gymnasium import spaces
    if isinstance(env.observation_space, spaces.Discrete):
        return lambda obs: int(obs)
    obs_dim = env.observation_space.shape[0]
    if bins is None:
        bins = 6
    low, high = env.observation_space.low, env.observation_space.high
    edges = [np.linspace(low[i], high[i], bins + 1) for i in range(obs_dim)]
    def to_key(obs):
        return tuple(int(np.clip(np.digitize(obs[i], edges[i]) - 1, 0, bins - 1)) for i in range(obs_dim))
    return to_key

def animate_policy(env, Q, steps=40, bins=None, delay=0.25):
    to_key = make_key_fn(env, bins=bins)
    obs, info = env.reset()
    for step in range(steps):
        clear_output(wait=True)
        display(HTML(env.render_html()))
        state = to_key(obs)
        action = max(range(env.action_space.n), key=lambda a: Q.get((state, a), 0.0))
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        print(f'step={step+1}  action={action}  reward={reward:.2f}  done={done}')
        time.sleep(delay)
        if done:
            clear_output(wait=True)
            display(HTML(env.render_html()))
            print(f'finished at step={step+1}, final reward={reward:.2f}')
            break

print('✅ RR environments ready!')


---
## Part 1 · Maze 2D — 2D Maze

Mirrors `Maze2D_emoji_en.html` on Rein Room.

**Game:** Grid maze. Agent starts at `(0, 0)` (bottom-left). Reach the goal for +10.

**Actions:** `0` = stay · `1` = up · `2` = down · `3` = left · `4` = right

**Levels** (`MAZE2D_LEVEL`):
| Level | Name | Grid | Goal | Notes |
|---|---|---|---|---|
| `0` | Open Field | 10×10 | (9, 9) | Default; larger state space, needs more episodes |
| `1` | Walled In | 5×5 | (4, 4) | Constrained space — path fills in faster, good for demos |

**`BINS`** sets how many buckets each axis is divided into for the Q-table.
Rule of thumb: set `BINS = grid_size` so each cell maps to exactly one bin.

**T3 task:** After training, look at the Q-table heatmap. Trace the arrow path from Start → Goal.


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
MAZE2D_LEVEL = 0     # 0 = Open Field (10x10, goal (9,9))  |  1 = Walled In (5x5, goal (4,4))
EPISODES     = 1000  # more episodes = more complete heatmap (try 500 for level=1)
ALPHA        = 0.5   # learning rate
GAMMA        = 0.95  # high gamma = agent values reaching the distant goal
EPSILON      = 0.3   # lower epsilon = more exploitation; arrows settle into a path earlier
BINS         = 10    # match grid_size: 10 for level=0, 5 for level=1
# ═══════════════════════════════════════════

env = Maze2DEnv(level=MAZE2D_LEVEL)
env.reset()
show_env(env)

In [ ]:
# Watch a manual route: right then up
env = Maze2DEnv()
env.reset()
animate_actions(env, [4,4,4,4,1,1,1,1], delay=0.25, stop_on_done=False)

In [ ]:
env = Maze2DEnv(level=MAZE2D_LEVEL)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Maze2D level={MAZE2D_LEVEL}, epsilon={EPSILON}', window=50)
plot_maze2d_qtable(Q, bins=BINS)


---
## Part 2 · Heli — Helicopter

Mirrors `heli_en.html` on Rein Room.

**Game:** Fly the helicopter through gaps in walls. Flap to rise; do nothing to fall.
- Survive each step: **+0.01**
- Pass through a wall gap: **+1**
- Hit wall or boundary: **−10**, episode ends

**Actions:** `0` = fall (gravity) · `1` = flap (move up)

**State (3 values, continuous):**
1. Helicopter Y position
2. Distance to next wall
3. Gap center Y

These are continuous — the agent must discretize them into buckets (`BINS`) to build a Q-table.

**Modes** (`HELI_MODE`) — how the gap position varies:
| Mode | Gap position | Difficulty |
|---|---|---|
| `fixed` | Always at center height | Easy — one fixed target height to learn |
| `small` | Center ± 100 px random each episode | Medium — gap shifts slightly |
| `large` | Center ± 200 px random each episode | Hard — gap can be near top or bottom |

**T4 task:** Run at least 50 episodes. Describe the reward curve trend — rising, flat, or noisy?


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
HELI_MODE = 'fixed'   # 'fixed'=easy  'small'=medium  'large'=hard
EPISODES  = 600       # run at least 50 for T4; 600 gives a clearer trend line
ALPHA     = 0.5       # learning rate
GAMMA     = 0.95      # high gamma: surviving more steps matters
EPSILON   = 0.25      # moderate exploration; too high = crashes a lot, too low = slow start
BINS      = 6         # discretize each of the 3 state dimensions into 6 buckets (6^3 = 216 states)
# ═══════════════════════════════════════════

env = HeliEnv(mode=HELI_MODE)
env.reset()
show_env(env)

In [ ]:
# Watch random actions in the game
env = HeliEnv(mode=HELI_MODE)
animate_random(env, steps=80, delay=0.06, stop_on_done=True)

In [ ]:
env = HeliEnv(mode=HELI_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Heli mode={HELI_MODE}, epsilon={EPSILON}', window=30)


---
## Part 3 · Fighter — Space Fighter  *(optional challenge)*

Mirrors `fighter_en.html` on Rein Room.

**Game:** Fly a fighter jet, dodge rocks, shoot them down. Hit 10 rocks to clear the stage.
- Bullet hits rock: **+10**
- Bullet misses (exits top): **−1**
- Rock hits player: **−100**, episode ends

**Actions:** `0` = none · `1` = move left · `2` = move right · `3` = shoot

**State (5 values, continuous):** playerX · rockX · rockY · rockVX · rockVY
5D continuous state — this is the most complex environment in the course.

**Modes** (`FIGHTER_MODE`) — rock behavior, ordered by difficulty:
| Mode | Rock behavior |
|---|---|
| `fixed` | Stationary at center top — just align and shoot |
| `randomX` | Random X position each episode — need to aim |
| `randomXY` | Random X and Y — varied distance |
| `falling` | Rock falls from top — moving target |
| `drifting` | Falls + drifts sideways — hardest to intercept |

No required task — explore freely. Try different modes and observe how the reward curve changes.


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
FIGHTER_MODE = 'fixed'  # difficulty order: 'fixed' < 'randomX' < 'randomXY' < 'falling' < 'drifting'
EPISODES     = 0        # how many episodes to train?
ALPHA        = 0.0      # learning rate
GAMMA        = 0.0      # discount factor
EPSILON      = 0.0      # exploration rate
BINS         = 0        # how many bins per state dimension?
# ═══════════════════════════════════════════

env = FighterEnv(mode=FIGHTER_MODE)
env.reset()
show_env(env)

In [ ]:
# Watch a fixed policy: shoot then wait
env = FighterEnv(mode=FIGHTER_MODE)
env.reset()
animate_actions(env, [3] + [0] * 30, delay=0.08, stop_on_done=True)

In [ ]:
env = FighterEnv(mode=FIGHTER_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Fighter mode={FIGHTER_MODE}, epsilon={EPSILON}', window=30)
